In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
import re
import shutil
from sklearn.utils import resample
from sklearn.model_selection import train_test_split


In [ ]:
!pip install kaggle

import os
os.environ['KAGGLE_USERNAME'] = "thisarasasmitha"
os.environ['KAGGLE_KEY'] = "c5c57b5a36e2bf59427fbe44a5e59805"


In [ ]:
# Download all datasets
# Fake only
!kaggle datasets download -d mahdimashayekhi/fake-news-detection-dataset -p /content/data --unzip
!kaggle datasets download -d saurabhshahane/fake-news-classification -p /content/data --unzip
!kaggle datasets download -d algord/fake-news -p /content/data --unzip
!kaggle datasets download -d ruchi798/source-based-news-classification -p /content/data --unzip
!kaggle datasets download -d mrisdal/fake-news -p /content/data --unzip

# Fake + Real (two files)
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p /content/data --unzip
!kaggle datasets download -d shawkyelgendy/fake-news-football -p /content/data --unzip
!kaggle datasets download -d emineyetm/fake-news-detection-datasets -p /content/data --unzip

# Real only
!kaggle datasets download -d sherazahmedfastian/true-news -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/mahdimashayekhi/fake-news-detection-dataset
License(s): MIT
  0% 0.00/11.2M [00:00<?, ?B/s]
100% 11.2M/11.2M [00:00<00:00, 993MB/s]
Dataset URL: https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification
License(s): Attribution 4.0 International (CC BY 4.0)
  0% 0.00/92.1M [00:00<?, ?B/s]
100% 92.1M/92.1M [00:00<00:00, 1.25GB/s]
Dataset URL: https://www.kaggle.com/datasets/algord/fake-news
License(s): CC0-1.0
  0% 0.00/1.68M [00:00<?, ?B/s]
100% 1.68M/1.68M [00:00<00:00, 540MB/s]
Dataset URL: https://www.kaggle.com/datasets/ruchi798/source-based-news-classification
License(s): CC0-1.0
  0% 0.00/2.97M [00:00<?, ?B/s]
100% 2.97M/2.97M [00:00<00:00, 533MB/s]
Dataset URL: https://www.kaggle.com/datasets/mrisdal/fake-news
License(s): CC0-1.0
  0% 0.00/19.4M [00:00<?, ?B/s]
100% 19.4M/19.4M [00:00<00:00, 956MB/s]
Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
  0% 0.

In [ ]:
def safe_read_csv(path, encoding='utf-8'):
    """Safely read CSV files with multiple encoding attempts"""
    try:
        return pd.read_csv(path, encoding=encoding)
    except UnicodeDecodeError:
        try:
            return pd.read_csv(path, encoding='latin-1')
        except:
            return pd.read_csv(path, encoding='ISO-8859-1')

def clean_text(text):
    """Comprehensive text cleaning"""
    if pd.isna(text):
        return ""

    text = str(text)
    # remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s\.\,\!\?\-\:]', ' ', text)
    # replace multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # remove URLs
    text = re.sub(r'http\S+', '', text)
    # remove user mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    return text.strip()

def standardize(df, title_col=None, text_col=None, label_col=None,
                label_mapping=None, fixed_label=None):
    """
    Convert dataset to schema: title, text, label
    """
    out = pd.DataFrame()

    # handle title
    if title_col and title_col in df.columns:
        out['title'] = df[title_col].fillna("")
    else:
        out['title'] = ""

    # handle text try multiple possible column names
    text_found = False
    possible_text_cols = [text_col, 'text', 'content', 'article', 'news', 'tweet']

    for col in possible_text_cols:
        if col and col in df.columns:
            out['text'] = df[col].fillna("")
            text_found = True
            break

    if not text_found:
        # use first column that looks like text
        for col in df.columns:
            if df[col].dtype == 'object' and col != title_col:
                out['text'] = df[col].fillna("")
                text_found = True
                break

    if not text_found:
        out['text'] = ""

    # handle labels
    if fixed_label is not None:
        out['label'] = fixed_label
    elif label_col and label_col in df.columns:
        if label_mapping:
            out['label'] = df[label_col].map(label_mapping)
        else:
            out['label'] = df[label_col]
    else:
        #  label from filename or dataset
        out['label'] = 0  # Default to fake

    return out

In [ ]:
datasets = []

In [ ]:
# -------------------------------
# News_dataset (Fake + True)
# -------------------------------
try:
    df_fake = safe_read_csv("/content/data/News _dataset/Fake.csv")
    df_true = safe_read_csv("/content/data/News _dataset/True.csv")

    if df_fake is not None:
        fake_std = standardize(df_fake, title_col="title", text_col="text", fixed_label=0)
        datasets.append(fake_std)

    if df_true is not None:
        true_std = standardize(df_true, title_col="title", text_col="text", fixed_label=1)
        datasets.append(true_std)
except Exception as e:
    print(f"Error")

# -------------------------------
# Fake.csv (Fake only)
# -------------------------------
try:
    df = safe_read_csv("/content/data/Fake.csv")
    if df is not None:
        std_df = standardize(df, title_col="title", text_col="text", fixed_label=0)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# FakeNewsNet.csv
# -------------------------------
try:
    df = safe_read_csv("/content/data/FakeNewsNet.csv")
    if df is not None:
        mapping = {0:0, 1:1}
        std_df = standardize(df, title_col="title", text_col="news_url",
                           label_col="real", label_mapping=mapping)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# 4. True.csv (Real only)
# -------------------------------
try:
    df = safe_read_csv("/content/data/True.csv")
    if df is not None:
        std_df = standardize(df, title_col="title", text_col="text", fixed_label=1)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# 5. WELFake_Dataset.csv
# -------------------------------
try:
    df = safe_read_csv("/content/data/WELFake_Dataset.csv")
    if df is not None:
        mapping = {0:0, 1:1}
        std_df = standardize(df, title_col="title", text_col="text",
                           label_col="label", label_mapping=mapping)
        datasets.append(std_df)
except Exception as e:
   print(f"Error")

# -------------------------------
# 6. fake.csv (tweets)
# -------------------------------
try:
    df = safe_read_csv("/content/data/fake.csv")
    if df is not None:
        std_df = standardize(df, title_col=None, text_col="tweet", fixed_label=0)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# 7. fake_news_dataset.csv
# -------------------------------
try:
    df = safe_read_csv("/content/data/fake_news_dataset.csv")
    if df is not None:
        mapping = {"fake":0, "Fake":0, "FAKE":0, "real":1, "Real":1, "REAL":1}
        std_df = standardize(df, title_col="title", text_col="text",
                           label_col="label", label_mapping=mapping)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# 8. news_articles.csv
# -------------------------------
try:
    df = safe_read_csv("/content/data/news_articles.csv")
    if df is not None:
        mapping = {"fake":0, "Fake":0, "FAKE":0, "real":1, "Real":1, "REAL":1}
        std_df = standardize(df, title_col="title", text_col="text",
                           label_col="label", label_mapping=mapping)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

# -------------------------------
# 9. real.csv (tweets)
# -------------------------------
try:
    df = safe_read_csv("/content/data/real.csv")
    if df is not None:
        std_df = standardize(df, title_col=None, text_col="tweet", fixed_label=1)
        datasets.append(std_df)
except Exception as e:
    print(f"Error")

print(f"Loaded {len(datasets)} datasets")

# merge all datasets
if datasets:
    all_data = pd.concat(datasets, ignore_index=True)

    # Add unique ID
    all_data.reset_index(inplace=True)
    all_data.rename(columns={'index':'id'}, inplace=True)

    print("Initial dataset info:")
    print(f"Shape: {all_data.shape}")
    print(f"Label distribution:\n{all_data['label'].value_counts()}")
else:
    raise ValueError("No datasets were successfully loaded!")

# data cleaning
def enhanced_data_cleaning(df):
    """Comprehensive data cleaning and preprocessing"""

    # remove rows with missing text
    df = df.dropna(subset=['text']).copy()

    # clean text
    df['clean_text'] = df['text'].apply(clean_text)

    # remove very short texts (< 50 characters)
    df = df[df['clean_text'].str.len() > 50].copy()

    # remove very long texts to avoid memory issues
    df = df[df['clean_text'].str.len() < 10000].copy()

    df = df.drop_duplicates(subset=['clean_text']).copy()

    # labels are integers (0 or 1)
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0, 1])].copy()

    return df

print("Cleaning data...")
all_data_cleaned = enhanced_data_cleaning(all_data)

print("After cleaning:")
print(f"Shape: {all_data_cleaned.shape}")
print(f"Label distribution:\n{all_data_cleaned['label'].value_counts()}")

# Balance the dataset
def balance_dataset(df):
    fake_news = df[df['label'] == 0]
    real_news = df[df['label'] == 1]

    print(f"Before balancing - Fake: {len(fake_news)}, Real: {len(real_news)}")

    # undersample the majority class
    if len(fake_news) > len(real_news):
        fake_news = resample(fake_news,
                           n_samples=len(real_news),
                           random_state=42,
                           replace=False)
    else:
        real_news = resample(real_news,
                           n_samples=len(fake_news),
                           random_state=42,
                           replace=False)

    balanced_data = pd.concat([fake_news, real_news])
    balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"After balancing - Fake: {len(fake_news)}, Real: {len(real_news)}")

    return balanced_data

print("Balancing dataset...")
balanced_data = balance_dataset(all_data_cleaned)

# Use clean_text as main text column
balanced_data['text'] = balanced_data['clean_text']
final_data = balanced_data[['id', 'title', 'text', 'label']].copy()

print("Final dataset info:")
print(f"Total samples: {len(final_data)}")
print(f"Label distribution:\n{final_data['label'].value_counts()}")

Loaded 10 datasets
Initial dataset info:
Shape: (249090, 4)
Label distribution:
label
1.0    129995
0.0    119094
Name: count, dtype: int64
Cleaning data...
After cleaning:
Shape: (138778, 5)
Label distribution:
label
1    76801
0    61977
Name: count, dtype: int64
Balancing dataset...
Before balancing - Fake: 61977, Real: 76801
After balancing - Fake: 61977, Real: 61977
Final dataset info:
Total samples: 123954
Label distribution:
label
1    61977
0    61977
Name: count, dtype: int64


In [ ]:
# Save the improved dataset
final_data.to_csv("/content/improved_fake_news_dataset2.csv", index=False)

# Copy to Google Drive
dst_folder = "/content/drive/MyDrive/"
dst = os.path.join(dst_folder, "improved_fake_news_dataset2.csv")
shutil.copy("/content/improved_fake_news_dataset2.csv", dst)
print(f" Improved dataset saved to: {dst}")

✅ Improved dataset saved to: /content/drive/MyDrive/improved_fake_news_dataset2.csv
